# Seleksi Kolom & Filtering Data Boolean

Kemampuan memilih (*selecting*) kolom dan menyaring (*filtering*) baris data berdasarkan kriteria tertentu adalah keterampilan paling esensial dalam analisis data sehari-hari.

## 1. Menyiapkan Dataset

In [1]:
import pandas as pd
import numpy as np

# Menyiapkan dataset demografi wilayah (440 baris) yang self-contained & representatif
np.random.seed(42)
n_samples = 440

nama_wilayah_list = [
    'Los_Angeles', 'Cook', 'Harris', 'San_Diego', 'Orange', 'Maricopa', 'Dallas', 
    'Miami_Dade', 'King', 'Clark', 'Wayne', 'Tarrant', 'Bexar', 'Santa_Clara', 
    'Wayne', 'Alameda', 'Middlesex', 'Hennepin', 'Sacramento', 'Travis', 'Fulton'
]
klasifikasi_list = ['West', 'Midwest', 'South', 'Northeast']

df = pd.DataFrame({
    'Nomor Identitas Wilayah': np.arange(1, n_samples + 1),
    'Nama Wilayah': [nama_wilayah_list[i % len(nama_wilayah_list)] for i in range(n_samples)],
    'Kode Wilayah': ['CA', 'IL', 'TX', 'CA', 'CA', 'AZ', 'TX', 'FL', 'WA', 'NV'][0:n_samples] if n_samples <= 10 else ['CA', 'IL', 'TX', 'CA', 'CA', 'AZ', 'TX', 'FL', 'WA', 'NV'] * (n_samples // 10) + ['CA'] * (n_samples % 10),
    'Luas Wilayah (mil2)': np.random.randint(15, 4500, size=n_samples),
    'Jumlah Penduduk (jiwa)': np.random.randint(100000, 3000000, size=n_samples),
    'Persen Penduduk 18-34th': np.round(np.random.uniform(16.0, 45.0, size=n_samples), 2),
    'Persen Penduduk >64th': np.round(np.random.uniform(5.0, 25.0, size=n_samples), 2),
    'Jumlah Dokter': np.random.randint(50, 15000, size=n_samples),
    'Jumlah Kejadian Kriminal': np.random.randint(500, 500000, size=n_samples),
    'Persen Lulusan Sarjana/Diploma IV': np.round(np.random.uniform(10.0, 50.0, size=n_samples), 2),
    'Persen Pengangguran': np.round(np.random.uniform(2.5, 15.0, size=n_samples), 2),
    'Pendapatan Perkapita': np.random.randint(9000, 38000, size=n_samples),
    'Klasifikasi Wilayah': np.random.choice(klasifikasi_list, size=n_samples, p=[0.35, 0.25, 0.25, 0.15])
})

print(f"Dataset berhasil dimuat: {df.shape[0]} baris × {df.shape[1]} kolom")

Dataset berhasil dimuat: 440 baris × 13 kolom


## 2. Seleksi Kolom (Series vs DataFrame)
- `df['Nama Kolom']`: Mengambil 1 kolom sebagai **Series**.
- `df[['Kolom1', 'Kolom2']]`: Mengambil beberapa kolom sekaligus sebagai **DataFrame**.

In [2]:
# 1. Mengambil satu kolom (Series)
wilayah_series = df['Nama Wilayah']
print("Tipe 1 Kolom:", type(wilayah_series))
print(wilayah_series.head(3))

# 2. Mengambil subset beberapa kolom (DataFrame)
subset_df = df[['Nama Wilayah', 'Jumlah Penduduk (jiwa)', 'Pendapatan Perkapita']]
print("\nTipe Banyak Kolom:", type(subset_df))
subset_df.head(3)

Tipe 1 Kolom: <class 'pandas.Series'>
0    Los_Angeles
1           Cook
2         Harris
Name: Nama Wilayah, dtype: str

Tipe Banyak Kolom: <class 'pandas.DataFrame'>


,Nama Wilayah,Jumlah Penduduk (jiwa),Pendapatan Perkapita
0,Los_Angeles,1139458,12664
1,Cook,2417336,28873
2,Harris,2863642,22050


## 3. Filter Baris Menggunakan Boolean Indexing
Kita membuat kondisi logika Boolean yang menghasilkan `True` atau `False` pada setiap baris.

In [3]:
# Filter 1: Wilayah dengan Penduduk > 1.5 Juta Jiwa
populasi_tinggi = df[df['Jumlah Penduduk (jiwa)'] > 1500000]
print(f"Jumlah wilayah berpopulasi > 1.5 juta: {len(populasi_tinggi)}")
populasi_tinggi[['Nama Wilayah', 'Jumlah Penduduk (jiwa)', 'Klasifikasi Wilayah']].head()

Jumlah wilayah berpopulasi > 1.5 juta: 215


,Nama Wilayah,Jumlah Penduduk (jiwa),Klasifikasi Wilayah
1,Cook,2417336,Midwest
2,Harris,2863642,South
3,San_Diego,1909891,Midwest
4,Orange,2519862,Midwest
6,Dallas,2486488,South


## 4. Filter dengan Multi Kondisi (`&` untuk AND, `|` untuk OR)
> ⚠️ **Catatan Penting**: Di Pandas, setiap kondisi logika harus diapit tanda kurung `(kondisi1) & (kondisi2)`.

In [4]:
# Filter: Wilayah 'West' DENGAN Pendapatan Perkapita > 25.000
filter_multi = df[(df['Klasifikasi Wilayah'] == 'West') & (df['Pendapatan Perkapita'] > 25000)]
print(f"Ditemukan {len(filter_multi)} wilayah yang memenuhi kriteria:")
filter_multi[['Nama Wilayah', 'Kode Wilayah', 'Pendapatan Perkapita', 'Klasifikasi Wilayah']].head()

Ditemukan 51 wilayah yang memenuhi kriteria:


,Nama Wilayah,Kode Wilayah,Pendapatan Perkapita,Klasifikasi Wilayah
16,Middlesex,TX,27287,West
21,Los_Angeles,IL,29544,West
36,Alameda,TX,27919,West
39,Sacramento,NV,35400,West
47,Maricopa,FL,28356,West


## 5. Metode Praktis: `isin()` dan `query()`
- `isin([...])`: Memfilter baris yang nilainya cocok dengan salah satu anggota daftar.
- `query("...")`: Menyaring data dengan sintaksis ekspresi string yang sangat bersih.

In [5]:
# Menggunakan isin()
df_west_midwest = df[df['Klasifikasi Wilayah'].isin(['West', 'Midwest'])]
print("Jumlah wilayah West & Midwest:", len(df_west_midwest))

# Menggunakan query()
df_hasil_query = df.query("`Pendapatan Perkapita` > 30000 and `Persen Pengangguran` < 5.0")
print("Wilayah pendapatan tinggi dengan pengangguran rendah:", len(df_hasil_query))
df_hasil_query[['Nama Wilayah', 'Pendapatan Perkapita', 'Persen Pengangguran']]

Jumlah wilayah West & Midwest: 249
Wilayah pendapatan tinggi dengan pengangguran rendah: 23


,Nama Wilayah,Pendapatan Perkapita,Persen Pengangguran
15,Alameda,34675,2.53
45,San_Diego,32255,4.31
76,Santa_Clara,36798,4.00
87,San_Diego,33335,4.86
120,Alameda,31724,4.12
135,Clark,31742,2.54
137,Tarrant,33703,4.83
150,San_Diego,31077,4.07
153,Dallas,31544,3.15
157,Wayne,34917,3.57
